# 第 12 章 · Local、Docker 与 Sandbox 环境

**这一章你会得到什么**：理解“同一个 Agent 换不同执行后端”是怎么做到的——环境工厂 + 统一 `execute` 接口。**本章不需要安装 docker**。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/environments/__init__.py` **L8–33** — `_ENVIRONMENT_MAPPING` + `get_environment(_class)` 工厂
- `src/minisweagent/environments/docker.py` **L1–45** — DockerEnvironment：常驻容器 + `docker exec`
- `src/minisweagent/environments/local.py` **L24–43** — 对照：LocalEnvironment 的同名 `execute` 接口

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 实验 1：环境工厂注册表

所有可用环境都登记在 `_ENVIRONMENT_MAPPING` 里，按名字动态加载。

In [ ]:
from minisweagent.environments import _ENVIRONMENT_MAPPING, get_environment_class, get_environment
for name, path in _ENVIRONMENT_MAPPING.items():
    print(f"{name:15} -> {path}")

## 实验 2：按名字拿到类，并用工厂造一个 local 环境

In [ ]:
cls = get_environment_class("local")
print("local ->", cls.__name__)
env = get_environment({"cwd": "/tmp"}, default_type="local")
print("造出的环境:", type(env).__name__)
print("跑一条:", env.execute({"command": "pwd"})["output"].strip())

## 观察点：Docker 环境的“持久容器”模式

看 `docker.py`（下面）：它先 `docker run -d ... sleep 2h` 起一个**常驻容器**，记住 `container_id`，
之后每条命令用 `docker exec` 进去跑。为什么不每条命令起一个新容器？因为启动容器很贵，
而且文件改动需要在**同一个容器**里累积。`LocalEnvironment` 和 `DockerEnvironment` 提供相同的 `execute`，
所以 Agent 完全无感——这正是第 2 章“可插拔沙箱”的兑现。

In [ ]:
show_source("src/minisweagent/environments/docker.py", 1, 45)

## 动手：给一个假的 `spec` 观察工厂的报错

调 `get_environment_class("no_such_env")`，看它抛什么错、错误信息里有没有列出可用环境。
（工厂对“未知类型”的处理，是一个好 harness 的可用性细节。）

In [ ]:
try:
    get_environment_class("no_such_env")
except ValueError as e:
    print(str(e)[:200])

## 闭卷检查
1. Agent 换执行后端时，靠的是什么保持不变？
2. Docker 环境为什么用常驻容器 + `docker exec`？
3. 环境工厂如何由字符串定位到类？